# Поиск и обезличивание персональных данных (PII NER)

Датасет для обучения: [redmadrobot-rnd/pii_train](https://huggingface.co/datasets/redmadrobot-rnd/pii_train): 17 137 русских текстов, где у каждого слова есть метка. Оценка — на **отдельном валидационном датасете** (раздел 8).

**Задача** — научить модель находить в тексте персональные данные (ФИО, адреса, телефоны, паспорта…) и заменять их, чтобы текст стал обезличенным.

Это задача **NER (Named Entity Recognition)**: мы классифицируем **каждое слово**, а не весь текст целиком, как было в IMDB.

План:
1. Загрузка и разбор данных
2. EDA: какие сущности есть, как выглядят, где шум
3. Метрика: entity-level F1
4. Подготовка данных для трансформера (субтокены и выравнивание меток)
5. Дообучение `cointegrated/rubert-tiny2` на всём `pii_train`
6. Обезличивание: маски `[FIRST_NAME]` и фейковые данные
7. Регулярки — бейзлайн для сравнения
8. Оценка на отдельном валидационном датасете и разбор ошибок

In [1]:
import json, re, random
from collections import Counter

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset, Dataset
from transformers import (AutoTokenizer, AutoModelForTokenClassification,
                          DataCollatorForTokenClassification, Trainer, TrainingArguments, pipeline)

device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

/Users/dashanalivayko/Desktop/Data Science/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device: mps


## 1. Загрузка данных

В датасете три колонки:
- `text` — исходный текст,
- `tokens` — текст, уже разрезанный на слова,
- `ner_tags` — метка для каждого слова.

`tokens` и `ner_tags` хранятся как **JSON-строки** (`'["отчество", "—", ...]'`), поэтому превращаем их в настоящие списки через `json.loads`.

Загрузку оформляем функцией `load_pii` — той же функцией потом загрузим и валидационный датасет. Она принимает имя датасета на Hugging Face или путь к файлу `.csv` / `.parquet` / `.json` / `.jsonl`.

In [2]:
def load_pii(source, split="train"):
    if source.endswith(".csv"):
        d = pd.read_csv(source)
    elif source.endswith(".parquet"):
        d = pd.read_parquet(source)
    elif source.endswith((".json", ".jsonl")):
        d = pd.read_json(source, lines=source.endswith(".jsonl"))
    else:
        d = load_dataset(source)[split].to_pandas()
    parse = lambda x: json.loads(x) if isinstance(x, str) else list(x)
    d["tokens"] = d["tokens"].apply(parse)
    d["tags"] = d["ner_tags"].apply(parse)
    return d[["tokens", "tags"]].reset_index(drop=True)

train_df = load_pii("redmadrobot-rnd/pii_train")
print(train_df.shape)
print("Длина tokens == длина tags во всех строках:", (train_df.tokens.str.len() == train_df.tags.str.len()).all())
train_df.head()

(17137, 2)
Длина tokens == длина tags во всех строках: True


,tokens,tags
0,"[отчество, —, Ермилович]","[O, O, B-MIDDLE_NAME]"
1,"[Пункт, приёма, на, ПРОСП, ., ДРУЖБЫ, ,, д, .,...","[O, O, O, B-STREET, I-STREET, I-STREET, O, B-H..."
2,"[Получатель:, макаров, Родион, Евгеньевич, ., ...","[O, B-LAST_NAME, B-FIRST_NAME, B-MIDDLE_NAME, ..."
3,"[Привет, ,, АГАТА, !, Ваш, запрос, о, замене, ...","[O, O, B-FIRST_NAME, O, O, O, O, O, O, O, O, O..."
4,"[Передайте, информацию, Родионова, пульхерия, ...","[O, O, B-LAST_NAME, B-FIRST_NAME, B-MIDDLE_NAM..."


### Формат меток BIO

Посмотрим на один пример слово за словом:
- `B-XXX` (Begin) — **первое** слово сущности типа XXX,
- `I-XXX` (Inside) — **продолжение** той же сущности,
- `O` (Outside) — обычное слово, не персональные данные.

Например, телефон `+7 929 861 66 51` — это пять слов: `B-PHONE I-PHONE I-PHONE I-PHONE I-PHONE`. Благодаря B/I модель понимает, где одна сущность закончилась и началась другая (два имени подряд — это `B-FIRST_NAME B-FIRST_NAME`, а не одно длинное).

In [3]:
def show(tokens, tags):
    return pd.DataFrame({"token": tokens, "tag": tags}).T

show(*train_df.loc[1, ["tokens", "tags"]])

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17
token,Пункт,приёма,на,ПРОСП,.,ДРУЖБЫ,",",д,.,176,.,Контакт:,+7,929,861,66,51,.
tag,O,O,O,B-STREET,I-STREET,I-STREET,O,B-HOUSE,I-HOUSE,I-HOUSE,O,O,B-PHONE,I-PHONE,I-PHONE,I-PHONE,I-PHONE,O


## 2. EDA

### Какие сущности есть и сколько их
Считаем только `B-` метки — так одна сущность считается один раз, сколько бы слов в ней ни было.

In [4]:
ent_counts = Counter(t[2:] for tags in train_df.tags for t in tags if t.startswith("B-"))
ent_counts = pd.Series(ent_counts).sort_values(ascending=False)
print("Типов сущностей:", len(ent_counts))
ent_counts

Типов сущностей: 21


FIRST_NAME           3876
LAST_NAME            3361
CITY                 2669
PASSPORT             2540
MIDDLE_NAME          2344
STREET               2190
PHONE                2134
URL                  2090
COUNTRY              1999
HOUSE                1906
EMAIL                1755
DISTRICT             1696
DRIVER_LICENSE       1688
REGION               1650
BIRTH_CERTIFICATE    1590
MILITARY_ID          1442
INN                  1163
CREDIT_CARD           989
IP_ADDRESS            939
SNILS                 924
OMS                   742
dtype: int64

In [5]:
lengths = train_df.tokens.str.len()
print("Слов в тексте: среднее %.1f, медиана %d, максимум %d" % (lengths.mean(), lengths.median(), lengths.max()))
print("Текстов вообще без персональных данных:", (train_df.tags.apply(lambda t: set(t) == {"O"})).sum())

Слов в тексте: среднее 22.9, медиана 19, максимум 388
Текстов вообще без персональных данных: 1237


### Как выглядят значения
Достанем из разметки сами сущности и посмотрим по несколько примеров на класс. Это важно, чтобы понять, **что можно поймать правилами, а что нет**.

In [6]:
def get_entities(tags):
    # BIO -> список (start, end, type), end не включительно
    ents, start, typ = [], None, None
    for i, t in enumerate(list(tags) + ["O"]):
        if typ is not None and not (t.startswith("I-") and t[2:] == typ):
            ents.append((start, i, typ))
            start, typ = None, None
        if t.startswith("B-") or (t.startswith("I-") and typ is None):
            start, typ = i, t[2:]
    return ents

examples = {}
for tokens, tags in zip(train_df.tokens, train_df.tags):
    for s, e, typ in get_entities(tags):
        examples.setdefault(typ, []).append(" ".join(tokens[s:e]))

random.seed(0)
for typ in ent_counts.index:
    print(f"{typ:18}", random.sample(examples[typ], 5))

FIRST_NAME         ['Людмила', 'Mark', 'Ювеналий', 'Назар', 'валентина']
LAST_NAME          ['Константинов', 'СмиРНоВ', 'КУЗНЕЦОВ', 'Новиков', 'Муравьев']
CITY               ['Самара', 'Казань', 'Воловець', 'Воронеж', 'КУРСК']
PASSPORT           ['123456', '098765', '123456', '112233', '123456']
MIDDLE_NAME        ['Дмитриевна;', 'Антиповичем', 'Юрьевич', 'ВАЛЕРИЕВНА', 'вячеславович']
STREET             ['бул . Просвещения', 'пр . Садовый', 'ул . Садовая', 'ул . Загородная', 'порто-франко']
PHONE              ['84951234567', '8-916-192-69-21', '929 865 57 79', '3', '+79221234567']
URL                ['https://apple . com/products', 'http://ortiz . com', 'https://lamoda . ru/download/report . pdf', 'https://booking . com', 'https://github . com/user/607']
COUNTRY            ['алЖИР', 'России', 'люксембург', 'Черногория', 'браЗилИЯ']
HOUSE              ['14 , квартира 3', 'д . 44', '10', 'д . 15 , кв . 4', 'д . 10 , кв . 5']
EMAIL              ['sofya . johnson@zoho . com', 'l . sergeev@

**Что видно:**
- Имена и адреса бывают в **любом регистре** (`бЕШиКТАш`, `АГАТА`, `макаров`) — регистр нельзя использовать как подсказку.
- `PASSPORT`, `DRIVER_LICENSE`, `INN`, `SNILS`, `OMS` — это **просто цифры**. `445566` может быть и паспортом, и правами. Отличить их можно только по **контексту** («паспорт серии…», «ВУ №…»). Это главный аргумент за ML-модель, которая читает соседние слова.
- Номера карт синтетические (`4111 - 2222 - 3333 - 4444`) — проверка контрольной суммы (алгоритм Луна) тут не помогла бы.
- Пунктуация отделена пробелами: `alert@monitor . com`, `192 . 168 . 1 . 1`.

### Шум в разметке
Разметка не идеальна. Например, тут паспорт и IBAN размечены как `O`:

In [7]:
show(*train_df.loc[3, ["tokens", "tags"]]).iloc[:, 20:30]

,20,21,22,23,24,25,26,27,28,29
token,копию,паспорта,48,58,940478,и,ваш,IBAN,RU52FAXZ4005593190566,для
tag,O,O,O,O,O,O,O,O,O,O


Такие ошибки разметки немного путают модель при обучении и немного «штрафуют» даже идеальную модель при оценке. Это нормально для реальных данных — просто держим в голове, что 100% не получится.

## 3. Метрика: entity-level F1

Метрику определяем сразу, хотя понадобится она в разделах 7–8.

Accuracy по словам тут обманчива: ~80% слов имеют метку `O`, и модель, которая всегда говорит `O`, получит accuracy 80%, не найдя ничего.

Поэтому считаем качество **по целым сущностям**: сущность найдена правильно, только если совпали **и границы, и тип**. Телефон из 5 слов, где модель нашла 4, — это ошибка.

- **Precision** — из того, что модель назвала персональными данными, какая доля правда ими была (нет ли ложных тревог).
- **Recall** — из всех настоящих персональных данных, какую долю модель нашла (ничего ли не пропустили).
- **F1** — среднее гармоническое этих двух.

Для обезличивания **recall важнее**: пропущенный паспорт — это утечка, а лишняя замена слова — мелкое неудобство.

Обычно эту метрику считают библиотекой `seqeval`, но мы напишем её сами — так понятнее, что происходит.

In [8]:
def entity_report(true_seqs, pred_seqs):
    tp, fp, fn = Counter(), Counter(), Counter()
    for t, p in zip(true_seqs, pred_seqs):
        T, P = set(get_entities(t)), set(get_entities(p))
        for e in T & P: tp[e[2]] += 1   # нашли правильно
        for e in P - T: fp[e[2]] += 1   # ложная тревога
        for e in T - P: fn[e[2]] += 1   # пропустили

    def prf(tp_, fp_, fn_):
        p = tp_ / (tp_ + fp_) if tp_ + fp_ else 0.0
        r = tp_ / (tp_ + fn_) if tp_ + fn_ else 0.0
        return p, r, (2 * p * r / (p + r) if p + r else 0.0)

    rows = [(typ, *prf(tp[typ], fp[typ], fn[typ]), tp[typ] + fn[typ])
            for typ in sorted(set(tp) | set(fp) | set(fn))]
    rows.append(("MICRO AVG", *prf(sum(tp.values()), sum(fp.values()), sum(fn.values())), sum(tp.values()) + sum(fn.values())))
    return pd.DataFrame(rows, columns=["entity", "precision", "recall", "f1", "support"]).set_index("entity").round(3)

## 4. Подготовка данных для трансформера

### Субтокены
Трансформер не работает со словами напрямую: токенизатор режет их на **субтокены** — кусочки из своего словаря. Редкие слова распадаются на много кусочков:

In [9]:
model_name = "cointegrated/rubert-tiny2"
tokenizer = AutoTokenizer.from_pretrained(model_name)

for w in ["Москва", "бЕШиКТАш", "Афанасьевна", "RU52FAXZ4005593190566"]:
    print(f"{w:24} -> {tokenizer.tokenize(w)}")

Москва                   -> ['Москва']
бЕШиКТАш                 -> ['б', '##Е', '##Ш', '##и', '##К', '##ТА', '##ш']
Афанасьевна              -> ['Афанась', '##евна']
RU52FAXZ4005593190566    -> ['RU', '##52', '##FA', '##X', '##Z', '##400', '##55', '##9', '##31', '##90', '##5', '##66']


### Выравнивание меток
Метки у нас **на слово**, а модель предсказывает **на субтокен**. Стандартный приём:
- первый субтокен слова получает метку слова,
- остальные субтокены и служебные токены `[CLS]`/`[SEP]` получают `-100` — это специальное значение «не учитывать в loss».

`is_split_into_words=True` говорит токенизатору, что текст уже разрезан на слова, а `word_ids()` показывает, из какого слова пришёл каждый субтокен.

Ещё модели нужны числа вместо строк, поэтому делаем словари `label2id` / `id2label`.

In [10]:
labels = ["O"] + sorted({t for tags in train_df.tags for t in tags} - {"O"})
label2id = {l: i for i, l in enumerate(labels)}
id2label = {i: l for l, i in label2id.items()}
print(len(labels), "меток:", labels[:7], "...")

def tokenize_and_align(batch):
    enc = tokenizer(batch["tokens"], is_split_into_words=True, truncation=True, max_length=512)
    all_labels = []
    for i, tags in enumerate(batch["tags"]):
        word_ids, prev, lab = enc.word_ids(i), None, []
        for w in word_ids:
            if w is None or w == prev:
                lab.append(-100)                # служебный токен или не первый кусочек слова
            else:
                lab.append(label2id[tags[w]])   # первый кусочек слова
            prev = w
        all_labels.append(lab)
    enc["labels"] = all_labels
    return enc

def to_hf(d):
    return Dataset.from_pandas(d[["tokens", "tags"]]).map(tokenize_and_align, batched=True, remove_columns=["tokens", "tags"])

train_ds = to_hf(train_df)

43 меток: ['O', 'B-BIRTH_CERTIFICATE', 'B-CITY', 'B-COUNTRY', 'B-CREDIT_CARD', 'B-DISTRICT', 'B-DRIVER_LICENSE'] ...


Map: 100%|██████████| 17137/17137 [00:01<00:00, 16219.74 examples/s]


Проверим на одном примере, что выравнивание правильное:

In [11]:
ex = train_ds[0]
pd.DataFrame({
    "subtoken": tokenizer.convert_ids_to_tokens(ex["input_ids"]),
    "label": [id2label.get(l, "-100") for l in ex["labels"]],
}).T

,0,1,2,3,4,5,6,7
subtoken,[CLS],от,##чество,[UNK],Ерм,##илов,##ич,[SEP]
label,-100,O,-100,O,B-MIDDLE_NAME,-100,-100,-100


## 5. Обучение

**Модель.** `cointegrated/rubert-tiny2` — маленький русский BERT (~29M параметров). Берём его первым, потому что он учится за минуты даже на ноутбуке. `AutoModelForTokenClassification` добавляет поверх BERT линейный слой, который для каждого субтокена выдаёт вероятности 43 меток.

**Fine-tuning** — модель уже «знает русский язык» после предобучения на огромном корпусе; мы лишь доучиваем её на нашей задаче.

Учим на **всём** `pii_train`: качество проверим на отдельном валидационном датасете (раздел 8).

**Параметры:**
- `learning_rate=1e-4` — для маленьких моделей обычно берут чуть больше, чем стандартные `2e-5…5e-5`,
- `num_train_epochs=3` — 3 прохода по данным. Во время обучения мы не видим качество на валидации, поэтому не ставим много эпох, чтобы не переобучиться,
- `DataCollatorForTokenClassification` — дополняет тексты в батче до одной длины (padding), и метки тоже (значением `-100`).

In [12]:
model = AutoModelForTokenClassification.from_pretrained(
    model_name, num_labels=len(labels), id2label=id2label, label2id=label2id)

args = TrainingArguments(
    output_dir="pii_checkpoints",
    learning_rate=1e-4,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="no",
    save_strategy="no",
    logging_steps=100,
    report_to="none",
    seed=42,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    processing_class=tokenizer,
    data_collator=DataCollatorForTokenClassification(tokenizer),
)
trainer.train()

Loading weights: 100%|██████████| 53/53 [00:00<00:00, 10329.36it/s]
[transformers] BertForTokenClassification LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architect

Step,Training Loss
100,1.057088
200,0.431289
300,0.275687
400,0.204086
500,0.163088
600,0.126916
700,0.104678
800,0.095575
900,0.085734
1000,0.080721


TrainOutput(global_step=1608, training_loss=0.18746112563197886, metrics={'train_runtime': 644.9693, 'train_samples_per_second': 79.711, 'train_steps_per_second': 2.493, 'total_flos': 97144630316160.0, 'train_loss': 0.18746112563197886, 'epoch': 3.0})

Как читать лог выше: **Training Loss** — ошибка модели на обучающих данных, она должна падать. Если к концу она почти перестала меняться, модель выучила всё, что могла, при этих настройках.

Сохраняем модель, чтобы не переобучать каждый раз:

In [13]:
trainer.save_model("pii_rubert_tiny2")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  8.47it/s]


## 6. Обезличивание

Применяем модель к **обычному тексту**. Тут есть две ловушки.

**1. Разбиение на слова.** Модель училась на словах, разрезанных определённым образом: точки, запятые и скобки — всегда отдельные слова (`г . Казань`, `192 . 168 . 1 . 1`, `+7 ( 917 ) 555-12-34`). Стандартный `pipeline` режет текст по-другому (например, `+7` → `+`, `7`), и сущности рассыпаются на куски: телефон `+7 (917) 555-12-34` превращается в PHONE `+` и PHONE `7 (917) 555-12-34`. Поэтому режем текст сами — функцией `split_words`, так же, как в `pii_train`, — и запоминаем, где в строке каждое слово (`start`/`end`), чтобы потом заменить его.

**2. Ложные срабатывания.** Модель иногда «находит» EMAIL / URL / PHONE там, где их нет — особенно в коде, SQL и рядах чисел (`SET @ date3` → EMAIL, `121 112 55 48 103` → PHONE). Поэтому каждую такую сущность проверяем на правдоподобие (`RULES`): склеиваем её слова без пробелов (`anna . petrova@mail . ru` → `anna.petrova@mail.ru`) и проверяем формат:
- **EMAIL** — должен иметь вид `что-то@домен.зона`,
- **URL** — должен содержать `http:` / `www.` или домен вида `site.ru`,
- **PHONE** — от 10 до 15 цифр (15 — максимум по международному стандарту).

Не прошла проверку — сущность отбрасываем. Модель находит кандидатов, правила отсеивают мусор. Эффект правил на метрике замерим в разделе 8.

### Декодирование Витерби: согласованные метки

Обычно для каждого слова берут метку с максимальной вероятностью (`argmax`) — **независимо от соседей**. Из-за этого модель иногда выдаёт невозможные последовательности:

```
ул        .   Баумана
B-STREET  O   I-STREET     ← после O не может идти I-STREET
```

Такая улица распадается на два обрывка, и ни один не засчитывается правильным. На бенчмарке таких невозможных переходов 471.

**Алгоритм Витерби** выбирает лучшую последовательность меток **целиком**, но только среди допустимых:
- `I-X` можно ставить только после `B-X` или `I-X`,
- текст не может начинаться с `I-X`.

Он перебирает не все варианты (их 43^длина), а умно: идёт по словам слева направо и для каждой метки помнит лучший путь, который в неё ведёт (динамическое программирование). Переобучать модель не нужно — работаем с теми же вероятностями.

In [ ]:
# TRANS[i, j] = 0, если после метки i можно поставить метку j, иначе -inf (запрещено)
n_labels = len(labels)
TRANS = np.zeros((n_labels, n_labels))
for j, lj in enumerate(labels):
    if lj.startswith("I-"):
        for i, li in enumerate(labels):
            if li not in ("B-" + lj[2:], "I-" + lj[2:]):
                TRANS[i, j] = -np.inf
START = np.array([-np.inf if l.startswith("I-") else 0.0 for l in labels])   # нельзя начинать с I-

def viterbi(log_probs):
    # log_probs: (число слов, число меток) — логарифмы вероятностей модели
    # возвращает лучшую ДОПУСТИМУЮ последовательность меток
    score = START + log_probs[0]                   # лучший счёт пути, заканчивающегося каждой меткой
    back = np.zeros(log_probs.shape, dtype=int)    # откуда пришли в эту метку
    for t in range(1, len(log_probs)):
        cand = score[:, None] + TRANS              # (из какой метки, в какую)
        back[t] = cand.argmax(0)
        score = cand.max(0) + log_probs[t]
    path = [int(score.argmax())]                   # идём назад от лучшей последней метки
    for t in range(len(log_probs) - 1, 0, -1):
        path.append(int(back[t, path[-1]]))
    return [labels[i] for i in path[::-1]]

# пример: argmax дал бы B-STREET, O, I-STREET
lp = np.log(np.full((3, n_labels), 1e-4))
lp[0, label2id["B-STREET"]] = np.log(0.9)
lp[1, label2id["O"]], lp[1, label2id["I-STREET"]] = np.log(0.55), np.log(0.4)
lp[2, label2id["I-STREET"]] = np.log(0.9)
print("argmax: ", [labels[i] for i in lp.argmax(-1)])
print("viterbi:", viterbi(lp))

In [ ]:
ner_model = AutoModelForTokenClassification.from_pretrained("pii_rubert_tiny2").to(device).eval()

RULES = {
    "EMAIL": lambda s: re.fullmatch(r"[\w.+\-]+@[\w\-]+(?:\.[\w\-]+)*\.[A-Za-zА-Яа-я]{2,}", s) is not None,
    "URL":   lambda s: re.search(r"(?:https?:|www\.)|[\w\-]\.[A-Za-zА-Яа-я]{2,}(?:[/:?#]|$)", s) is not None,
    "PHONE": lambda s: 10 <= len(re.sub(r"\D", "", s)) <= 15,
}

SPLIT = r"[.,;!?()\[\]{}«»\"]"   # как в pii_train: эти знаки всегда отдельные слова ("192 . 168 . 1 . 1", "А . В .")
EDGE = ":'"                      # а эти — только на краях слова (внутри "https://" и IPv6 оставляем)

def split_words(text):
    # возвращает позиции слов в строке: [(start, end), ...]
    words = []
    for m in re.finditer(rf"{SPLIT}|[^\s.,;!?()\[\]{{}}«»\"]+", text):
        s, e = m.start(), m.end()
        while e - s > 1 and text[s] in EDGE:
            words.append((s, s + 1)); s += 1
        tail = []
        while e - s > 1 and text[e - 1] in EDGE:
            tail.append((e - 1, e)); e -= 1
        words.append((s, e))
        words += tail[::-1]
    return words

def find_entities(text, min_score=0.5):
    spans = split_words(text)
    words = [text[s:e] for s, e in spans]
    enc = tokenizer(words, is_split_into_words=True, truncation=True, max_length=512, return_tensors="pt").to(device)
    with torch.no_grad():
        log_probs = ner_model(**enc).logits[0].log_softmax(-1).cpu().numpy()

    # берём первый субтокен каждого слова (как при обучении); слова за пределами 512 субтокенов не дошли до модели
    first = {}
    for k, w in enumerate(enc.word_ids()):
        if w is not None and w not in first:
            first[w] = k
    idx = sorted(first)
    word_lp = log_probs[[first[w] for w in idx]]

    # score слова = уверенность, что это вообще персональные данные: 1 - P(O)
    # (не вероятность выбранной метки: Витерби может выбрать I-SNILS с P=0.07, если модель колебалась между I-SNILS и I-PHONE)
    tags, scores = ["O"] * len(words), [0.0] * len(words)
    for w, tag, row in zip(idx, viterbi(word_lp), word_lp):
        tags[w], scores[w] = tag, 1 - float(np.exp(row[label2id["O"]]))

    ents = []
    for s, e, typ in get_entities(tags):
        score = sum(scores[s:e]) / (e - s)          # средняя уверенность по словам сущности
        if score < min_score:
            continue
        if typ in RULES and not RULES[typ]("".join(words[s:e])):
            continue                                 # не похоже на настоящий EMAIL / URL / PHONE
        ents.append({"entity_group": typ, "score": score, "start": spans[s][0], "end": spans[e - 1][1]})
    return ents

sample = ("Здравствуйте! Меня зовут Петрова Анна Сергеевна, живу в г. Казань, ул. Баумана, д. 12. "
          "Паспорт 9204 123456, СНИЛС 123-456-789 01. Телефон +7 (917) 555-12-34, почта anna.petrova@mail.ru")

for e in find_entities(sample):
    print(f"{e['entity_group']:15} {e['score']:.2f}  {sample[e['start']:e['end']]!r}")

Два способа обезличить:
1. **Маска** — заменить на тип: `[FIRST_NAME]`. Просто, надёжно, сразу видно, что было удалено.
2. **Псевдонимизация** — заменить на правдоподобные фейковые данные через `Faker('ru_RU')`. Текст остаётся естественным, его удобно использовать дальше (например, отдавать другой модели).
   Одинаковые значения заменяем одинаково (словарь `cache`) — иначе «Анна» в начале и в конце письма станут разными людьми.

Замены идём делать **с конца строки к началу** — тогда позиции ещё не обработанных сущностей не сдвигаются.

In [ ]:
from faker import Faker
fake = Faker("ru_RU")

FAKERS = {
    "FIRST_NAME": fake.first_name, "LAST_NAME": fake.last_name, "MIDDLE_NAME": fake.middle_name,
    "CITY": fake.city_name, "STREET": fake.street_name, "HOUSE": fake.building_number,
    "COUNTRY": fake.country, "REGION": fake.region,
    "PHONE": fake.phone_number, "EMAIL": fake.free_email, "URL": fake.url, "IP_ADDRESS": fake.ipv4,
    "CREDIT_CARD": fake.credit_card_number, "INN": fake.individuals_inn,
    "PASSPORT": lambda: fake.numerify("#### ######"), "SNILS": lambda: fake.numerify("###-###-### ##"),
    "OMS": lambda: fake.numerify("################"), "DRIVER_LICENSE": lambda: fake.numerify("## ## ######"),
}

def anonymize(text, mode="mask", min_score=0.5):
    cache = {}
    ents = find_entities(text, min_score)
    for e in sorted(ents, key=lambda e: e["start"], reverse=True):
        typ, original = e["entity_group"], text[e["start"]:e["end"]]
        if mode == "fake" and typ in FAKERS:
            repl = cache.setdefault((typ, original.lower()), FAKERS[typ]())
        else:
            repl = f"[{typ}]"
        text = text[:e["start"]] + repl + text[e["end"]:]
    return text

print(anonymize(sample), "\n")
print(anonymize(sample, mode="fake"))

Проверим на текстах другого стиля — без пробелов перед пунктуацией, как в живой переписке:

In [ ]:
texts = [
    "Иван Петрович, переведите оплату на карту 4276 1600 1234 5678 и пришлите чек на ivanov.ip@yandex.ru",
    "Сотрудник Смирнов А.В., ИНН 772012345678, водительское удостоверение 77 12 345678.",
    "Сервер 10.0.0.15 недоступен, пишите в чат https://t.me/devops_team",
    "Встреча в Новосибирске, Красный проспект 25, в пятницу. Звонить Ольге: 89131234567.",
    "SELECT * FROM users WHERE id = 1; SET @date3 = DATEADD(month, -1, GETDATE())",   # тут ПД нет
]
for t in texts:
    print("ДО:   ", t)
    print("ПОСЛЕ:", anonymize(t))
    print()

## 7. Бейзлайн: регулярные выражения

Чтобы понять, сколько на самом деле даёт нейросеть, сравним её с простыми правилами. Регулярки хорошо подходят для данных **с жёстким форматом**: email, URL, IP, телефон, карта.

Как это работает:
1. Склеиваем слова через пробел в строку (поэтому в регулярках `' ?\. ?'` — точка с необязательными пробелами вокруг).
2. Ищем совпадения регулярками → получаем позиции символов.
3. Переводим позиции символов обратно в метки слов (`B-`/`I-`).

Ожидаем, что правила неплохо найдут email/URL/IP, но не найдут имена, адреса и документы, а регулярка телефона будет путать его с ИНН и ОМС (это тоже 10–16 цифр подряд).

In [17]:
DOT = r" ?\. ?"
TLD = r"(?:ru|com|org|net|de|io|gov|edu|info|su|by|kz|ua|uk|рф)"
PATTERNS = [  # порядок важен: более специфичные — раньше
    ("EMAIL",       rf"[\w\-]+(?:{DOT}[\w\-]+)*@[\w\-]+(?:{DOT}[\w\-]+)+"),
    ("URL",         rf"(?:https?://)?[\w\-]+(?:{DOT}[\w\-]+)*{DOT}{TLD}\b(?:/\S*)?"),
    ("CREDIT_CARD", r"(?:(?:\d{4}|(?:\* ?){4})[ \-]*){3}\d{4}"),
    ("IP_ADDRESS",  rf"\b\d{{1,3}}(?:{DOT}\d{{1,3}}){{3}}(?::\d+)?|\b[0-9a-fA-F]{{1,4}}(?::[0-9a-fA-F]{{0,4}}){{2,7}}\b"),
    ("PHONE",       r"(?:\+\d{1,3}|\b8)?[ \-]*\(? ?\d{3} ?\)?[ \-]*\d{3}[ \-]*\d{2}[ \-]*\d{2}\b"),
]

def regex_tag(tokens):
    text, offsets, pos = "", [], 0
    for tok in tokens:                       # запоминаем, где в строке начинается и кончается каждое слово
        offsets.append((pos, pos + len(tok)))
        text += tok + " "
        pos += len(tok) + 1

    tags = ["O"] * len(tokens)
    for typ, pattern in PATTERNS:
        for m in re.finditer(pattern, text):
            idx = [i for i, (s, e) in enumerate(offsets) if s < m.end() and e > m.start()]
            if idx and all(tags[i] == "O" for i in idx):     # не перезаписываем уже найденное
                tags[idx[0]] = "B-" + typ
                for i in idx[1:]:
                    tags[i] = "I-" + typ
    return tags

show(train_df.tokens[1], regex_tag(train_df.tokens[1]))

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17
token,Пункт,приёма,на,ПРОСП,.,ДРУЖБЫ,",",д,.,176,.,Контакт:,+7,929,861,66,51,.
tag,O,O,O,O,O,O,O,O,O,O,O,O,B-PHONE,I-PHONE,I-PHONE,I-PHONE,I-PHONE,O


## 8. Оценка на валидационном датасете

Валидация — [redmadrobot-rnd/pii_benchmark](https://huggingface.co/datasets/redmadrobot-rnd/pii_benchmark): 2 841 текст в том же формате, что `pii_train`, с теми же 21 типом сущностей. Ни один текст не совпадает с `pii_train`.

Загружаем его **только здесь**: ни EDA, ни обучение его не видели — поэтому оценка на нём честная.

In [18]:
VAL_SOURCE = "redmadrobot-rnd/pii_benchmark"

val_df = load_pii(VAL_SOURCE, split="test")   # в этом датасете есть только сплит "test"
print(val_df.shape)

# метки, которых не было в train, модель предсказать не может — предупреждаем и считаем их "O"
unknown = Counter(t for tags in val_df.tags for t in tags if t not in label2id)
if unknown:
    print("Метки, которых нет в train (заменены на O):", dict(unknown))
    val_df["tags"] = val_df.tags.apply(lambda tags: [t if t in label2id else "O" for t in tags])

(2841, 2)


Предсказываем метки моделью. Модель выдаёт предсказание для каждого субтокена; берём метку с максимальной вероятностью (`argmax`) и оставляем только позиции, где label не `-100`, — это ровно первые субтокены слов, то есть метки слов.

In [19]:
def logits_to_tags(logits, label_ids):
    pred_ids = logits.argmax(-1)
    true_seqs, pred_seqs = [], []
    for p_row, l_row in zip(pred_ids, label_ids):
        mask = l_row != -100
        true_seqs.append([id2label[i] for i in l_row[mask]])
        pred_seqs.append([id2label[i] for i in p_row[mask]])
    return true_seqs, pred_seqs

out = trainer.predict(to_hf(val_df))
val_true, val_pred = logits_to_tags(out.predictions, out.label_ids)
model_report = entity_report(val_true, val_pred)
model_report

Map: 100%|██████████| 2841/2841 [00:00<00:00, 3822.10 examples/s]
/Users/dashanalivayko/Desktop/Data Science/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


,precision,recall,f1,support
entity,,,,
BIRTH_CERTIFICATE,0.897,0.894,0.896,341
CITY,0.627,0.614,0.620,339
COUNTRY,0.690,0.736,0.712,242
CREDIT_CARD,0.870,0.892,0.881,203
DISTRICT,0.616,0.826,0.706,161
DRIVER_LICENSE,0.856,0.817,0.836,371
EMAIL,0.441,0.978,0.608,181
FIRST_NAME,0.781,0.773,0.777,488
HOUSE,0.681,0.798,0.734,163


Те же данные — регулярками:

In [20]:
regex_report = entity_report(val_df.tags, [regex_tag(t) for t in val_df.tokens])
regex_report

,precision,recall,f1,support
entity,,,,
BIRTH_CERTIFICATE,0.000,0.000,0.000,341
CITY,0.000,0.000,0.000,339
COUNTRY,0.000,0.000,0.000,242
CREDIT_CARD,0.524,0.852,0.649,203
DISTRICT,0.000,0.000,0.000,161
DRIVER_LICENSE,0.000,0.000,0.000,371
EMAIL,0.928,0.525,0.671,221
FIRST_NAME,0.000,0.000,0.000,499
HOUSE,0.000,0.000,0.000,163


Сравним по F1:

In [21]:
pd.DataFrame({"regex_f1": regex_report.f1, "model_f1": model_report.f1}).fillna(0).sort_values("model_f1")

,regex_f1,model_f1
entity,,
URL,0.567,0.542
EMAIL,0.671,0.608
STREET,0.000,0.614
CITY,0.000,0.620
PHONE,0.385,0.667
LAST_NAME,0.000,0.686
DISTRICT,0.000,0.706
REGION,0.000,0.706
COUNTRY,0.000,0.712


### Модель + правила

Применяем к предсказаниям модели те же правила `RULES` из раздела 6: EMAIL / URL / PHONE, которые не похожи на настоящие, превращаем в `O`. Больше всего это помогает на коде и рядах чисел в бенчмарке, где модель находила «почту» в `SET @ date3`.

Цена — немного recall у PHONE: неполные номера вроде `8 - 916 - 123 - 45` (меньше 10 цифр) тоже снимаются.

In [ ]:
def postprocess(tokens, tags):
    tags = list(tags)
    for s, e, typ in get_entities(tags):
        rule = RULES.get(typ)
        if rule and not rule("".join(tokens[s:e])):   # склеиваем слова без пробелов и проверяем
            tags[s:e] = ["O"] * (e - s)
    return tags

val_pred_pp = [postprocess(w, p) for w, p in zip(val_df.tokens, val_pred)]
pp_report = entity_report(val_true, val_pred_pp)

pd.DataFrame({"regex_f1": regex_report.f1, "model_f1": model_report.f1,
              "model+rules_f1": pp_report.f1}).fillna(0).sort_values("model+rules_f1")

### Модель + Витерби (+ правила)

Те же вероятности модели, но метки выбираем не `argmax`, а Витерби (раздел 6) — только допустимые последовательности. Больше всего это помогает многословным сущностям (улицы, регионы, телефоны), у которых больше мест, где цепочка может «порваться».

In [ ]:
def log_softmax(x):
    x = x - x.max(-1, keepdims=True)
    return x - np.log(np.exp(x).sum(-1, keepdims=True))

# out.predictions — логиты модели из ячейки выше; берём строки первых субтокенов слов (label != -100)
val_pred_vit = [viterbi(log_softmax(logits[l_row != -100]))
                for logits, l_row in zip(out.predictions, out.label_ids)]
vit_report = entity_report(val_true, val_pred_vit)
vit_pp_report = entity_report(val_true, [postprocess(w, p) for w, p in zip(val_df.tokens, val_pred_vit)])

pd.DataFrame({"model_f1": model_report.f1, "model+rules_f1": pp_report.f1,
              "viterbi_f1": vit_report.f1, "viterbi+rules_f1": vit_pp_report.f1}).fillna(0).sort_values("viterbi+rules_f1")

### Разбор ошибок
Метрики говорят «сколько», а примеры — «почему». Покажем несколько текстов, где модель ошиблась, и только слова с расхождениями.

In [22]:
errors = [i for i, (t, p) in enumerate(zip(val_true, val_pred)) if t != p]
print(f"Текстов с ошибками: {len(errors)} из {len(val_true)}\n")

random.seed(1)
for i in random.sample(errors, min(6, len(errors))):
    tokens = val_df.tokens[i]
    print(" ".join(tokens)[:200])
    for w, t, p in zip(tokens, val_true[i], val_pred[i]):
        if t != p:
            print(f"    {w:25} правда: {t:20} модель: {p}")
    print()

Текстов с ошибками: 798 из 2841

Уважаемый пациент , для ускорения процесса регистрации в системе ЕГИСЗ , пожалуйста , укажите ваш СНИЛС : 555 - 111 - 222 33 .
    ЕГИСЗ                     правда: O                    модель: B-COUNTRY

Слушайте, я уже совсем вымоталась от всех этих формальностей. Сегодня мне пришлось снова искать своё свидетельство о рождении, а номер у него — IV-ZI 666390 Я уже три раза звонила в отдел кадров, но к
    666390                    правда: I-BIRTH_CERTIFICATE  модель: B-BIRTH_CERTIFICATE

Скажите , а электронный паспарт 9988 776655 можно предъявить при входе в ваше отделение ?
    9988                      правда: B-PASSPORT           модель: O
    776655                    правда: I-PASSPORT           модель: I-CREDIT_CARD

{ " warehouse " : { " addr " : пр-т Ленина , " bld " : стр . 7А , " link " : https://warehouse-system . net/stock } }
    стр                       правда: B-HOUSE              модель: B-STREET

Протокол № 42 заседания комиссии по р

Типичные ошибки, которые стоит искать в выводе выше:
- **путаница похожих классов** — `DRIVER_LICENSE` ↔ `PASSPORT`, `CITY` ↔ `REGION`, `FIRST_NAME` ↔ `LAST_NAME`;
- **границы** — модель нашла сущность, но захватила лишнее слово или потеряла одно;
- **ошибки разметки** — иногда модель права, а «правда» размечена неверно (как паспорт из раздела 2).

## Итоги и что дальше

- Регулярки неплохо справляются с данными жёсткого формата, но не понимают контекста и не находят имена, адреса и документы.
- Дообученный BERT читает контекст и может различать похожие по виду классы (паспорт / права / ИНН).
- Обезличивание = NER + замена найденных фрагментов по позициям.

**Ограничения:**
- Faker не склоняет слова: «Петровой» может замениться на «Иванов».
- Faker не учитывает пол: женское имя может замениться на мужское.
- Обучающие данные синтетические — на реальных текстах (письма, чаты, сканы документов) качество будет ниже.

**Как улучшить:**
1. Модель побольше: `deepvk/deberta-v1-base`, `DeepPavlov/rubert-base-cased`, `FacebookAI/xlm-roberta-base` — меняется только `model_name` (и `learning_rate` на `3e-5`).
2. Подобрать число эпох: обучить на 2, 3, 5 и сравнить F1 на валидации.
3. Гибрид: модель + регулярки для EMAIL/URL/IP (объединять найденное — это повышает recall).
4. Понизить `min_score` в `anonymize` — больше recall ценой ложных замен.
5. Сравнить с zero-shot моделью `urchade/gliner_multi_pii-v1`, которая ничего не обучала.